In [62]:
import re
%run monodromyfunctions.ipynb

class Pi1():
    #Established once and for all that each R,L letter applies a dehn twist which effects generators of the fundamental group as follows.
    #For reference, see Baker-Peterson, Character Varieties of Once-Punctured Torus Bundles with Tunnel Number 1 pg. 5-7
    monodromy_dict_R = {109 : 'm', 108 : 'l*m'}
    monodromy_dict_L = {109 : 'm*l', 108 : 'l'}
    
    #This dictionary sends both m and l to 1 and leaves t unchanged.
    abelianize_dict = {109 : '1', 108 : '1'}
    
    def __init__(self, monodromy):
        self.relator1 = self.make_relator1(monodromy)
        self.relator2 = self.make_relator2(monodromy)
        self.pi1 = self.make_pi1(monodromy, self.relator1, self.relator2) 
        self.homology = self.make_homology(monodromy)
      
    #Given a letter R or L applies the appropriate twist.
    def twist(self, twist:str, word:str):
        if twist == 'R':
            return word.translate(self.monodromy_dict_R)
        elif twist == 'L':
            return word.translate(self.monodromy_dict_L)
  
    #Apply the monodromy to the fundamental group to generate two relators. Composition reads right to left.
    #Eliminate a letter from the monodromy word after it's been applied until no letters remain.
    def make_relator1(self, monodromy):
        relator = 'm'
        while monodromy != '':
            relator = self.twist(monodromy[-1], relator)
            monodromy = monodromy[:-1]
        return relator
    
    def make_relator2(self, monodromy):
        relator = 'l'
        while monodromy != '':
            relator = self.twist(monodromy[-1], relator)
            monodromy = monodromy[:-1]
        return relator
    
    #Plug these relators into a group presentation which then gets input to the magma engine.
    def make_pi1(self, monodromy, relator1, relator2):
        return magma(f'FPGroup <m,l,t | t^-1*m*t={relator1}, t^-1*l*t = {relator2}>')
    
    
    #Fox-derivative product rule. Don't need to consider inverse functions because only t appears on the lhs with inverse, so we can manually deal with this case.
    
    def product_rule(self, der, word):
        der_indices = [m.start() for m in re.finditer(der, word)]
        
        if der_indices[0] == 0:
            derivative = '1'
        else:
            derivative = word[:der_indices[0]-1]
        
        for i in range(1, len(der_indices)):
            derivative = derivative + ' + ' + word[:der_indices[i]-1]
   
        return derivative
            
    
    def fox_derivative_rel1(self, der):
        if der == 't':
            return '-t^-1 + t^-1*m'
        elif der == 'l':
            return self.product_rule(der, self.relator1)
        else:
            return f'-t^-1 + {self.product_rule(der, self.relator1)}'
        
    def fox_derivative_rel2(self, der):
        if der == 't':
            return '-t^-1 + t^-1*l'
        elif der == 'm':
            return self.product_rule(der, self.relator2)
        else:
            return f'-t^-1 + {self.product_rule(der, self.relator2)}'
        
        
    #The integral smith normal form gives a matrix which is reduced via a sequence of row and column operations.
    #The resulting diagonal matrix gives the abelian relation for m and l on the first and second diagonals respectively.
    
    def make_homology(self, monodromy):
        mat = (wordtomatrix(monodromy) - matrix.identity(2)).smith_form()[0]
        self.hom_relator1 = mat[0,0]
        self.hom_relator2 = mat[1,1]
        
        G = AbelianGroup([mat[0,0], mat[1,1], 0])
        
        return G
    
    
        

In [70]:
G = Pi1('RLRL').homology; G

Multiplicative Abelian group isomorphic to C1 x C5 x Z

In [74]:
G.algebra(QQbar)

Algebra of Multiplicative Abelian group isomorphic to C1 x C5 x Z over Algebraic Field